# 7.1 问题、Mask 与相位

用合成音频建立音源分离的基本对象：source、mixture、STFT、ideal mask，以及复用 mixture phase 的重建限制，不依赖 MUSDB18-HQ 或任何其他素材。


## 兼容性提示

- 请使用前面章节已经建立的虚拟环境中运行本章 notebook（推荐 Python 3.11）。部分依赖包在其他版本上可能出现 `collections.Hashable` 等兼容性问题。
- 首次运行预训练模型时，工具会自动下载 checkpoint，需要一定时间，请耐心等待。


## 1. 环境准备


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# matplotlib/numba 缓存目录，用跨平台的系统临时目录
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "numba_cache"))

# 路径推断：从 cwd 向上找含 CODE/chapter07/_common 的目录；NOTEBOOK_DIR 指向 CODE/chapter07/
_p = Path.cwd()
while not (_p / "CODE" / "chapter07" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter07/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
NOTEBOOK_DIR = _p / "CODE" / "chapter07"
CODE_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

FIG_DIR = NOTEBOOK_DIR / "output_figures"
FIG_DIR.mkdir(exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR.relative_to(REPO_ROOT))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from chapter07._common.metrics import reconstruction_error, si_sdr
from chapter07._common.plotting import (
    FIGURE_SAVE_DPI,
    GRAY_IMAGE_CMAP,
    GRAY_MASK_CMAP,
    LINE_GRAYS,
    plot_matrix_grid,
    plot_waveforms,
)
from chapter07._common.spectrogram import amplitude_to_db
from chapter07._common.spectrogram import (
    apply_mask,
    ideal_binary_mask,
    ideal_ratio_mask,
    istft,
    stft,
)
from chapter07._common.synthesis import make_synthetic_mixture


## 2. 合成三个 source 并相加


In [ ]:
sr = 22050
sources = make_synthetic_mixture(sr=sr, duration=4.0, seed=7)

ordered = {
    "harmonic": sources["harmonic"],
    "bass": sources["bass"],
    "percussive": sources["percussive"],
    "mixture": sources["mixture"],
}
plot_waveforms(ordered, sr, FIG_DIR / "07_1_mixture_and_sources.png")
plt.show()

err = reconstruction_error(
    sources["mixture"],
    {name: sources[name] for name in ("harmonic", "bass", "percussive")},
)
print(f"relative reconstruction error: {err:.3e}")


## 3. 线性与对数频谱


In [ ]:
preview_spec = stft(sources["mixture"], n_fft=2048, hop_length=512)
preview_mag = np.abs(preview_spec)
preview_db = amplitude_to_db(preview_mag)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
axes[0].imshow(preview_mag, origin="lower", aspect="auto", cmap=GRAY_IMAGE_CMAP)
axes[0].set_title("线性幅度谱")
axes[1].imshow(preview_db, origin="lower", aspect="auto", cmap=GRAY_IMAGE_CMAP)
axes[1].set_title("对数幅度谱 (dB)")
for ax in axes:
    ax.set_xlabel("帧")
    ax.set_ylabel("频率 bin")
fig.tight_layout()
plt.show()


## 4. Ideal binary mask 与 ideal ratio mask


In [ ]:
n_fft = 2048
hop_length = 512

mixture_spec = stft(sources["mixture"], n_fft=n_fft, hop_length=hop_length)
source_specs = {
    name: stft(sources[name], n_fft=n_fft, hop_length=hop_length)
    for name in ("harmonic", "bass", "percussive")
}
source_mags = {name: np.abs(spec) for name, spec in source_specs.items()}
mag_list = list(source_mags.values())

target = "harmonic"
ibm = ideal_binary_mask(source_mags[target], mag_list)
irm = ideal_ratio_mask(source_mags[target], mag_list)

plot_matrix_grid(
    {
        "IBM：谐波源": ibm,
        "IRM：谐波源": irm,
    },
    FIG_DIR / "07_1_ideal_masks.png",
    cmap=GRAY_MASK_CMAP,
    vmin=0.0,
    vmax=1.0,
)
plt.show()

print("IRM range:", float(irm.min()), float(irm.max()))


## 5. 复用 mixture phase 的重建限制


In [ ]:
estimated_spec = apply_mask(mixture_spec, irm)
estimated_harmonic = istft(estimated_spec, hop_length=hop_length, length=len(sources[target]))

oracle_spec = source_specs[target]
oracle_harmonic = istft(oracle_spec, hop_length=hop_length, length=len(sources[target]))

phase_reuse_sisdr = si_sdr(sources[target], estimated_harmonic)
oracle_sisdr = si_sdr(sources[target], oracle_harmonic)
print(f"IRM + mixture phase SI-SDR: {phase_reuse_sisdr:.2f} dB")
print(f"Oracle complex STFT SI-SDR: {oracle_sisdr}")


In [ ]:
t = np.arange(len(sources[target])) / sr
start = int(0.9 * sr)
end = int(1.4 * sr)

fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t[start:end], sources[target][start:end], linewidth=0.9, color=LINE_GRAYS[0])
axes[0].set_title("参考谐波源")
axes[1].plot(t[start:end], estimated_harmonic[start:end], linewidth=0.9, color=LINE_GRAYS[1])
axes[1].set_title("IRM + 混合相位重建")
axes[2].plot(
    t[start:end],
    sources[target][start:end] - estimated_harmonic[start:end],
    linewidth=0.9,
    color=LINE_GRAYS[2],
)
axes[2].set_title("残差")
axes[2].set_xlabel("时间 (s)")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_1_phase_reuse_error.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
plt.show()


## 6. 小结

- mixture 是多个 sources 的叠加；stem taxonomy 决定了分离任务本身。
- ideal mask 是上限分析工具，不是实际模型。
- 只估计 magnitude 并复用 mixture phase 会留下相位和干扰源共同造成的残差。
